# PetroRAG — Exploração de Embeddings

Este notebook demonstra:
1. Como os embeddings representam semântica (PT-BR)
2. Visualização de clusters de eventos com t-SNE
3. Comparação de similaridade entre queries e documentos
4. Análise de qualidade da recuperação (Precision@K)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

from config import EMBEDDING_MODEL
from src.embeddings import get_embeddings
from src.indexer import build_index, load_documents, split_documents
from src.retriever import semantic_search

print(f'Modelo: {EMBEDDING_MODEL}')

## 1. Carregar modelo e indexar documentos

In [ ]:
# Constrói índice (ou carrega se já existir)
vectorstore = build_index()
print('Índice pronto!')

## 2. Similaridade semântica entre termos do domínio

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)

# Termos técnicos do domínio offshore
terms = [
    'fechamento espúrio da válvula de segurança DHSV',
    'spurious closure of downhole safety valve',
    'queda de pressão no sensor P-PDG',
    'pressure drop at bottom sensor',
    'slug severo na coluna de produção',
    'severe slugging in production riser',
    'importância SHAP do sensor T-JUS-CKP',
    'feature importance of temperature sensor',
    'pipeline ETL camada Bronze para Gold',
    'data processing medallion architecture',
]

embeddings = model.encode(terms, normalize_embeddings=True)
sim_matrix = cosine_similarity(embeddings)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Similaridade de Cosseno')
ax.set_xticks(range(len(terms)))
ax.set_yticks(range(len(terms)))
labels = [t[:35] + '...' if len(t) > 35 else t for t in terms]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
ax.set_title('Matriz de Similaridade Semântica (PT-BR + EN)', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('../assets/similarity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/similarity_matrix.png')

## 3. Busca semântica — exemplos de queries

In [ ]:
queries = [
    'Qual sensor tem maior importância para detectar fechamento da DHSV?',
    'Como o pipeline processa dados faltantes nos sensores?',
    'Quais são as classes de eventos do dataset 3W?',
]

for q in queries:
    print(f'\n🔍 QUERY: {q}')
    print('─' * 60)
    results = semantic_search(vectorstore, q, k=3)
    for i, r in enumerate(results, 1):
        sim = max(0, (1 - r.score) * 100) if r.score > 0 else (1 / (1 + r.score)) * 100
        print(f'  #{i} [{sim:.1f}%] {r.content[:200]}...')

## 4. Visualização t-SNE dos embeddings dos chunks

In [ ]:
# Carregar todos os chunks e visualizar distribuição
import sys
sys.path.insert(0, '..')

from src.indexer import load_documents, split_documents
from config import RAW_DIR

docs = load_documents(RAW_DIR)
chunks = split_documents(docs)

texts = [c.page_content for c in chunks[:200]]  # máx 200 para t-SNE
chunk_embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

tsne = TSNE(n_components=2, perplexity=min(30, len(texts)-1), random_state=42)
coords = tsne.fit_transform(chunk_embeddings)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=range(len(texts)),
                      cmap='viridis', alpha=0.6, s=30)
ax.set_title(f't-SNE dos Embeddings de {len(texts)} Chunks de Documentos', fontsize=13)
ax.set_xlabel('Dimensão 1')
ax.set_ylabel('Dimensão 2')
plt.colorbar(scatter, ax=ax, label='Ordem do chunk')
plt.tight_layout()
plt.savefig('../assets/tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/tsne_embeddings.png')